# Lab 01: Survey Data Recoding — Python for SAS/Stata Users

**WatSPEED Agentic AI Prep Hub | Stage 1 of 4**

---

## What Dataset Are We Working With?

We are using the **Kaggle AI Trust Insights Dataset** — 1,200 survey respondents answering questions about their demographic background, perceived AI risk, and trust in artificial intelligence.

| Column | Type | Values | SAS Equivalent |
|---|---|---|---|
| `Respondent_ID` | String | `RESP_00101` etc. | Character `$10.` |
| `Age_Group` | Categorical | `18-29`, `30-44`, `45-60`, `60+` | CLASS variable |
| `Education_Level` | Categorical | `High School`, `Bachelor's`, `Master's`, `PhD` | FORMAT labels |
| `Perceived_AI_Risk` | Likert (1–5) | 1=Very Low … 5=Extreme; **-9 = missing** | Missing = `.` |
| `High_AI_Trust` | Binary outcome | 1=High Trust, 0=Low Trust | Y variable |
| `Survey_Weight` | Float | e.g. `1.25` | WEIGHT statement |

## What Are We Doing and Why?

**Goal**: Recode raw survey data (handle missing values coded as `-9`, create binary flags) and compute a **population-weighted mean** — exactly what you'd do in SAS before running `PROC LOGISTIC`.

**Why this matters for Agentic AI**: Before an LLM agent can call your Python analysis tools, the data must be clean and validated. This lab shows the cleaning pipeline.

---

## ✏️ Your Tasks
1. Run the default code and inspect the output
2. **Change `recode_cutoff = 4` to `recode_cutoff = 3`** — does the high-risk count change?
3. **Add a new respondent** to the `data` list with your own values
4. **Change the missing value code** from `-9` to `-99` and add a respondent with that value

In [ ]:
# ============================================================
# Cell 1: Install & Import
# In Colab/Jupyter you can use real pandas here!
# ============================================================
import pandas as pd
import numpy as np

print('pandas version:', pd.__version__)

In [ ]:
# ============================================================
# Cell 2: Sample Dataset (AI Trust Insights, 1,200 respondents)
# In a real project you'd load this from CSV:
#   df = pd.read_csv('data/ai_trust_insights.csv')
# ============================================================
raw_data = [
    # id         age_group   education      ai_risk  tech_fam  ai_trust  weight
    {'id': 'RESP_01', 'age_group': '30-44', 'education': "Master's",   'ai_risk': 4,  'tech_fam': 5, 'ai_trust': 1, 'weight': 1.25},
    {'id': 'RESP_02', 'age_group': '18-29', 'education': "Bachelor's", 'ai_risk': -9, 'tech_fam': 4, 'ai_trust': 1, 'weight': 0.95},  # -9 = missing!
    {'id': 'RESP_03', 'age_group': '45-60', 'education': 'High School','ai_risk': 5,  'tech_fam': 2, 'ai_trust': 0, 'weight': 1.10},
    {'id': 'RESP_04', 'age_group': '60+',   'education': 'PhD',        'ai_risk': 4,  'tech_fam': 3, 'ai_trust': 0, 'weight': 1.40},
    {'id': 'RESP_05', 'age_group': '18-29', 'education': "Master's",   'ai_risk': 1,  'tech_fam': 5, 'ai_trust': 1, 'weight': 0.88},
    {'id': 'RESP_06', 'age_group': '30-44', 'education': 'High School','ai_risk': 3,  'tech_fam': 3, 'ai_trust': 0, 'weight': 1.05},
    {'id': 'RESP_07', 'age_group': '45-60', 'education': "Bachelor's", 'ai_risk': -9, 'tech_fam': 4, 'ai_trust': 0, 'weight': 1.15},  # -9 = missing!
    {'id': 'RESP_08', 'age_group': '60+',   'education': 'PhD',        'ai_risk': 5,  'tech_fam': 1, 'ai_trust': 0, 'weight': 1.50},
]

df = pd.DataFrame(raw_data)
print('=== RAW DATA (SAS PROC PRINT equivalent) ===')
print(df.to_string(index=False))

In [ ]:
# ============================================================
# Cell 3: Recode Missing Values & Create Binary Flag
# SAS equivalent:
#   DATA clean; SET raw;
#     IF ai_risk = -9 THEN ai_risk = .;   /* recode to missing */
#     high_risk_flag = (ai_risk >= 4);    /* binary flag */
#   RUN;
# ============================================================

# ✏️ EDIT THIS VALUE — try changing to 3 or 5 and re-running!
recode_cutoff = 4
missing_code  = -9   # SAS uses '.' for numeric missing; surveys often use -9 or -99

# Step 1: Replace -9 with NaN (Python's equivalent of SAS missing '.')
df['ai_risk_clean'] = df['ai_risk'].replace(missing_code, np.nan)

# Step 2: Create binary high-risk flag (1 if risk >= cutoff, 0 otherwise)
# Note: pandas comparison with NaN returns False automatically
df['high_risk_flag'] = (df['ai_risk_clean'] >= recode_cutoff).astype(int)

# Step 3: Mark valid rows (not missing)
df['valid'] = df['ai_risk_clean'].notna()

print(f'=== RECODED DATA (cutoff >= {recode_cutoff}) ===')
print(df[['id', 'age_group', 'ai_risk', 'ai_risk_clean', 'high_risk_flag', 'valid']].to_string(index=False))
print(f'\nMissing values found: {df["ai_risk_clean"].isna().sum()} out of {len(df)} respondents')
print(f'High-risk respondents (ai_risk >= {recode_cutoff}): {df["high_risk_flag"].sum()}')

In [ ]:
# ============================================================
# Cell 4: Population-Weighted Mean
# SAS equivalent:
#   PROC SURVEYMEANS DATA=clean;
#     WEIGHT survey_weight;
#     VAR ai_risk;
#   RUN;
# ============================================================

# Only use non-missing rows for the weighted mean
valid_df = df[df['valid']].copy()

# Weighted mean = sum(value * weight) / sum(weight)
weighted_mean = (valid_df['ai_risk_clean'] * valid_df['weight']).sum() / valid_df['weight'].sum()
unweighted_mean = valid_df['ai_risk_clean'].mean()

print('=== WEIGHTED SURVEY ANALYSIS ===')
print(f'Valid respondents (non-missing): {len(valid_df)}')
print(f'Unweighted Mean AI Risk:  {unweighted_mean:.3f} / 5.0')
print(f'Population Weighted Mean: {weighted_mean:.3f} / 5.0')
print(f'Difference (weighting effect): {weighted_mean - unweighted_mean:+.3f}')

In [ ]:
# ============================================================
# Cell 5: Pydantic Validation Schema
# This is what protects your AI agent pipeline from bad data.
# SAS analogy: PROC FORMAT + DATA step ERROR log checks
# ============================================================
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional

class SurveyRespondent(BaseModel):
    respondent_id: str
    age_group: str
    ai_risk_raw: int
    ai_risk_clean: Optional[float] = None
    survey_weight: float

    @field_validator('age_group')
    @classmethod
    def valid_age_group(cls, v):
        allowed = ['18-29', '30-44', '45-60', '60+']
        if v not in allowed:
            raise ValueError(f'age_group must be one of {allowed}')
        return v

    @field_validator('survey_weight')
    @classmethod
    def positive_weight(cls, v):
        if v <= 0:
            raise ValueError('survey_weight must be positive')
        return v

# ✏️ Try changing '18-29' to 'Under 18' — you should get a ValidationError!
try:
    r = SurveyRespondent(
        respondent_id='RESP_TEST',
        age_group='18-29',
        ai_risk_raw=4,
        ai_risk_clean=4.0,
        survey_weight=1.10
    )
    print('✅ Validation passed:', r)
except ValidationError as e:
    print('❌ Validation failed (expected):')
    print(e)